# 3.1 向量检索基础

构建VectorRetriever - 智能知识库Agent的语义检索引擎

## 项目定位
本实验的VectorRetriever被HybridSearcher和6.3最终项目直接使用

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client; client = get_client()
from agent_project import *
from agent_project.tools import create_default_registry
print(f"LLM: {client.name} | project modules loaded")


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, time
print("Loading BGE embedding model...")
model = SentenceTransformer("BAAI/bge-small-zh-v1.5")
dim = model.get_sentence_embedding_dimension()
print(f"Model loaded: dim={dim}")


In [ ]:
DOCS = [
    "Transformer架构使用Self-Attention机制处理序列数据",
    "向量数据库如FAISS和Milvus高效存储和检索高维向量",
    "RAG系统结合检索和生成来提高回答准确性和可追溯性",
    "BM25是经典的关键词检索算法基于TF-IDF改进用于精确匹配",
    "DeepSeek V4于2026年4月发布1.6T参数MoE架构SWE-bench80.6%",
    "Qwen3.7-Max支持1M上下文中文能力在国产模型中排名第一",
    "MCP协议是AI Agent标准化协议已捐赠Linux Foundation",
    "LoRA通过低秩矩阵分解实现高效模型微调仅训练0.1%参数",
    "混合检索将BM25和向量检索结合通过RRF融合结果提升召回率",
    "自反思评估自动检测检索质量触发降级策略保证最低质量",
]
embeddings = model.encode(DOCS, normalize_embeddings=True)
print(f"Knowledge base: {len(DOCS)} docs, vectors: {embeddings.shape}")


In [ ]:
import faiss
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
print(f"FAISS index: IndexFlatIP, {index.ntotal} vectors, {dim}d")

queries = ["如何提高搜索准确性?", "最新的国产AI模型有哪些?", "Agent标准协议是什么?"]
for q in queries:
    qv = model.encode([q], normalize_embeddings=True)
    D, I = index.search(qv.astype(np.float32), 3)
    print(f"\nQuery: {q}")
    for rank, (did, score) in enumerate(zip(I[0], D[0])):
        print(f"  #{rank+1} [{score:.4f}] {DOCS[did][:60]}...")


In [ ]:
# Performance benchmark
for n in [100, 1000, 5000]:
    v = np.random.randn(n, dim).astype(np.float32)
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    idx = faiss.IndexFlatIP(dim); idx.add(v)
    qv = np.random.randn(1, dim).astype(np.float32)
    t0 = time.time()
    for _ in range(100): idx.search(qv, 10)
    dt = (time.time()-t0)/100*1000
    print(f"  {n:>5} vectors: {dt:.2f}ms/query, {n*dim*4/1024/1024:.1f}MB")
